In [7]:
import gmsh
import numpy as np
from collections import defaultdict


class DimensionIDTracker:
    def __init__(self):
        self.entities = {}     # (dim, id) -> name
        self.parents = defaultdict(set)
        self.children = defaultdict(set)
        self.embeds = defaultdict(set)   # (3, vol_id) -> set((2, fault_id))
        self.physical_groups = {}        # name -> (dim, tag)

    # ------------------------------------------------------------
    # ---- Core entity registration
    # ------------------------------------------------------------
    def register(self, dim, tag, name=None, parents=None):
        key = (dim, tag)
        if name:
            self.entities[key] = name
            try:
                gmsh.model.setEntityName(dim, tag, name)
            except Exception:
                pass
        if parents:
            self.parents[key].update(parents)
            for p in parents:
                self.children[p].add(key)

    def get_name(self, dim, tag):
        return self.entities.get((dim, tag), gmsh.model.getEntityName(dim, tag))

    def create_physical_group(self, dim, entities, name):
        tag = gmsh.model.addPhysicalGroup(dim, [e[1] for e in entities])
        gmsh.model.setPhysicalName(dim, tag, name)
        self.physical_groups[name] = (dim, tag)
        return (dim, tag)

    # ------------------------------------------------------------
    # ---- Embedding management
    # ------------------------------------------------------------
   
    def embed_fault(self, fault_dimtags, volume_dimtags):
        """
        Embed fault surfaces into target volumes.
        """
        for vol in volume_dimtags:
            gmsh.model.mesh.embed(fault_dimtags[0][0],
                                  [f[1] for f in fault_dimtags],
                                  vol[0], vol[1])
            self.embeds[vol].update(fault_dimtags)
        gmsh.model.occ.synchronize()


    def get_embedded_faults(self, vol):
        """Return faults embedded in a volume."""
        return list(self.embeds.get(vol, []))

    # ------------------------------------------------------------
    # ---- Hierarchy-preserving rotation + embedding
    # ------------------------------------------------------------
    def rotate_group(self, dim_tags, center, axis, angle_deg, name_suffix="_rot"):
        """
        Rotate a group of entities (volumes, surfaces, faults, etc.),
        preserve hierarchy, name inheritance, and embeddings.
        """
        before = {d: set(gmsh.model.occ.getEntities(d)) for d in range(1, 4)}

        gmsh.model.occ.rotate(dim_tags, *center, *axis, np.deg2rad(angle_deg))
        gmsh.model.occ.synchronize()

        after = {d: set(gmsh.model.occ.getEntities(d)) for d in range(1, 4)}
        new_entities = [(d, t) for d in range(1, 4)
                        for (d, t) in after[d] - before[d]]

        orig_by_dim = {d: [e for e in dim_tags if e[0] == d] for d in range(1, 4)}

        # Register new entities with inherited names and hierarchy
        for new_dim, new_tag in new_entities:
            parent_candidates = orig_by_dim.get(new_dim, [])
            if parent_candidates:
                parent = parent_candidates[0]
                pname = self.get_name(*parent)
                new_name = f"{pname}{name_suffix}{int(angle_deg)}"
                self.register(new_dim, new_tag, new_name, parents=[parent])
                self.create_physical_group(new_dim, [(new_dim, new_tag)], new_name)
            else:
                # Generic fallback
                self.register(new_dim, new_tag,
                              f"entity_{new_dim}_{new_tag}{name_suffix}{int(angle_deg)}")
                self.create_physical_group(new_dim, [(new_dim, new_tag)],
                                           f"entity_{new_dim}_{new_tag}{name_suffix}{int(angle_deg)}")

        # ---- Re-embed faults into rotated volumes ----
        for vol, faults in list(self.embeds.items()):
            # Find rotated volume matching this one
            vname = self.get_name(*vol)
            rotated_vol = next(((d, t) for (d, t) in new_entities
                                if d == 3 and self.get_name(d, t).startswith(vname)), None)
            if rotated_vol:
                # Find matching rotated faults
                rotated_faults = []
                for f in faults:
                    fname = self.get_name(*f)
                    rotated = [e for e in new_entities
                               if e[0] == f[0] and self.get_name(*e).startswith(fname)]
                    rotated_faults.extend(rotated)
                if rotated_faults:
                    gmsh.model.mesh.embed(2, [f[1] for f in rotated_faults], 3, rotated_vol[1])
                    gmsh.model.occ.synchronize()
                    self.embeds[rotated_vol].update(rotated_faults)

        return new_entities

    # ------------------------------------------------------------
    # ---- Debug / inspection
    # ------------------------------------------------------------
    def describe(self):
        print("\n=== Entity Summary ===")
        for (dim, tag), name in self.entities.items():
            parents = [self.entities.get(p, str(p)) for p in self.parents.get((dim, tag), [])]
            children = [self.entities.get(c, str(c)) for c in self.children.get((dim, tag), [])]
            print(f"({dim},{tag}): {name}")
            if parents:
                print(f"  parents: {parents}")
            if children:
                print(f"  children: {children}")
        if self.embeds:
            print("\n=== Embeddings ===")
            for vol, faults in self.embeds.items():
                vname = self.get_name(*vol)
                fnames = [self.get_name(*f) for f in faults]
                print(f"  Volume {vname} embeds faults: {fnames}")


In [9]:
gmsh.finalize()

In [12]:
import gmsh

gmsh.initialize()
gmsh.model.add("embed_fault_demo")

tracker = DimensionIDTracker()

# --- Create one volume ---
vol = gmsh.model.occ.addBox(0, 0, 0, 1, 1, 1)
tracker.register(3, vol, "MainVolume")
gmsh.model.occ.synchronize()
# --- Create a fault plane cutting through it ---
fault_loop = gmsh.model.occ.addRectangle(0.5, 0, 0, 0.01, 1)
fault_surf = gmsh.model.occ.addPlaneSurface([fault_loop])
gmsh.model.occ.synchronize()
tracker.register(2, fault_surf, "FaultA", parents=[(3, vol)])

# --- Embed fault ---
tracker.embed_fault([(2, fault_surf)], [(3, vol)])

gmsh.model.occ.synchronize()

# --- Rotate everything together ---
rotated = tracker.rotate_group(
    [(3, vol), (2, fault_surf)],
    center=(0.5, 0.5, 0.5),
    axis=(0, 0, 1),
    angle_deg=30
)

tracker.describe()

gmsh.write("embedded_fault_rotated.msh")
gmsh.finalize()



=== Entity Summary ===
(3,1): MainVolume
  children: ['FaultA']
(2,8): FaultA
  parents: ['MainVolume']
  children: ['FaultA_rot30', 'FaultA_rot30', 'FaultA_rot30', 'FaultA_rot30', 'FaultA_rot30', 'FaultA_rot30']
(1,28): entity_1_28_rot30
(1,31): entity_1_31_rot30
(1,34): entity_1_34_rot30
(1,18): entity_1_18_rot30
(1,21): entity_1_21_rot30
(1,24): entity_1_24_rot30
(1,30): entity_1_30_rot30
(1,27): entity_1_27_rot30
(1,33): entity_1_33_rot30
(1,36): entity_1_36_rot30
(1,20): entity_1_20_rot30
(1,17): entity_1_17_rot30
(1,23): entity_1_23_rot30
(1,26): entity_1_26_rot30
(1,32): entity_1_32_rot30
(1,29): entity_1_29_rot30
(1,35): entity_1_35_rot30
(1,19): entity_1_19_rot30
(1,25): entity_1_25_rot30
(1,22): entity_1_22_rot30
(2,14): FaultA_rot30
  parents: ['FaultA']
(2,10): FaultA_rot30
  parents: ['FaultA']
(2,13): FaultA_rot30
  parents: ['FaultA']
(2,9): FaultA_rot30
  parents: ['FaultA']
(2,12): FaultA_rot30
  parents: ['FaultA']
(2,11): FaultA_rot30
  parents: ['FaultA']

=== Embe

In [ ]:
=== Entity Summary ===
(3,1): MainVolume
  children: ['FaultA']
(2,8): FaultA
  parents: ['MainVolume']
  children: ['FaultA_rot30', 'FaultA_rot30', 'FaultA_rot30', 'FaultA_rot30', 'FaultA_rot30', 'FaultA_rot30']
(1,28): entity_1_28_rot30
(1,31): entity_1_31_rot30
(1,34): entity_1_34_rot30
(1,18): entity_1_18_rot30
(1,21): entity_1_21_rot30
(1,24): entity_1_24_rot30
(1,30): entity_1_30_rot30
(1,27): entity_1_27_rot30
(1,33): entity_1_33_rot30
(1,36): entity_1_36_rot30
(1,20): entity_1_20_rot30
(1,17): entity_1_17_rot30
(1,23): entity_1_23_rot30
(1,26): entity_1_26_rot30
(1,32): entity_1_32_rot30
(1,29): entity_1_29_rot30
(1,35): entity_1_35_rot30
(1,19): entity_1_19_rot30
(1,25): entity_1_25_rot30
(1,22): entity_1_22_rot30
(2,14): FaultA_rot30
  parents: ['FaultA']
(2,10): FaultA_rot30
  parents: ['FaultA']
(2,13): FaultA_rot30
  parents: ['FaultA']
(2,9): FaultA_rot30
  parents: ['FaultA']
(2,12): FaultA_rot30
  parents: ['FaultA']
(2,11): FaultA_rot30
  parents: ['FaultA']

=== Embeddings ===
  Volume MainVolume embeds faults: ['FaultA']
Info    : Writing 'embedded_fault_rotated.msh'...
Info    : Done writing 'embedded_fault_rotated.msh'